In [2]:
#import
import ConnectionConfig as cc
debugging_mode=True

In [3]:
#config
cc.setupEnvironment()
spark = cc.startLocalCluster("AnalyseVragenS2",4)
spark.getActiveSession()

Environment variables are set...


In [4]:
#Dimensies en fact inladen
dateDimDf= spark.read.format("delta").load("./delta/DATE_DIM")
rainDimDf= spark.read.format("delta").load("./delta/RAIN_DIM")
seasonDimDf= spark.read.format("delta").load("./delta/SEASON_DIM")
userDimDf = spark.read.format("delta").load("./spark-warehouse/dimuser")
treasureTypeDimDf = spark.read.format("delta").load("./spark-warehouse/dimtreasuretype")
treasureFoundFactDf = spark.read.format("delta").load("./delta/FACT_TREASURE_FOUND")

dateDimDf.createOrReplaceTempView("dimDate")
rainDimDf.createOrReplaceTempView("dimRain")
seasonDimDf.createOrReplaceTempView("dimSeason")
userDimDf.createOrReplaceTempView("dimUser")
treasureTypeDimDf.createOrReplaceTempView("dimTreasureType")
treasureFoundFactDf.createOrReplaceTempView("factTreasureFound")

In [17]:
#  Wat is de invloed van het type user op de duur van de treasurehunt?
spark.sql("""
    SELECT
        dU.experiencelevel,
        COUNT(*) as number_of_hunts,
        ROUND(AVG(fTF.duration) / 60, 2) as avg_duration_minutes
    FROM factTreasureFound fTF
    JOIN dimUser dU ON dU.userSurKey = fTF.UserSurKey
    WHERE dU.experiencelevel IS NOT NULL
    GROUP BY dU.experiencelevel
    ORDER BY dU.experiencelevel
""").show(100)

+---------------+---------------+--------------------+
|experiencelevel|number_of_hunts|avg_duration_minutes|
+---------------+---------------+--------------------+
|        Amateur|        1044695|               70.03|
|         Pirate|         654752|               70.03|
|   Professional|          90251|               70.11|
+---------------+---------------+--------------------+



In [24]:
#  Vinden users de cache gemiddeld sneller in de regen?
spark.sql("""
    SELECT
        dR.RainCode as condition,
        COUNT(*) as hunts,
        ROUND(AVG(fTF.duration) / 60, 1) as avg_minutes
    FROM factTreasureFound fTF
    JOIN dimRain dR ON dR.RainSurKey = fTF.RainSurKey
    WHERE dR.RainCode IN ('RAIN', 'NORAIN')
    GROUP BY dR.RainCode
    ORDER BY dR.RainCode DESC
""").show()

+---------+-----+-----------+
|condition|hunts|avg_minutes|
+---------+-----+-----------+
|     RAIN|  114|      114.7|
|   NORAIN|  276|       66.3|
+---------+-----+-----------+



In [38]:
#  Zoeken beginnende users gemiddeld naar grotere caches (aantal stages)?
spark.sql("""
          SELECT COUNT(*) as hunts,
                 dTT.size
          FROM factTreasureFound fTF
                   JOIN dimUser dU ON dU.userSurKey = fTF.UserSurKey
                   JOIN dimTreasureType dTT ON dTT.TreasureTypeSurKey = fTF.TreasureTypeSurKey
          WHERE dU.experiencelevel = 'Amateur'
          GROUP BY dTT.size
          order by hunts DESC
          """).show()

+------+----+
| hunts|size|
+------+----+
|391128|   1|
|141908|   5|
|103275|   4|
|102607|   6|
|101685|   7|
| 61037|   8|
| 41384|   9|
| 41331|   3|
| 40675|   2|
| 19665|  10|
+------+----+

